In [ ]:
from setup import *

## Unit 4.4: Subgroup Performance and Model Reporting

### DAIR-3 Workshop, Summer 2026
<b>Presented by Suraj Rampure (rampure@umich.edu)</b>

---

### Overview

- We have several ways to measure overall model performance:
    - MSE and RMSE,
    - MAE,
    - $R^2$,
    - cross-validated performance,
    - final held-out test performance.
- A regulator, scientist, or program administrator may also ask:
    - Does the model perform differently for different groups?
- How might we report all of these figures to non-technical stakeholders?

## Subgroup performance

---

### Subgroup performance

- One aspect of model performance a regulator may care about is subgroup behavior.
- For each subgroup, ask:
    - How many observations are in this group?
    - What is the group's MAE or RMSE?
    - Does the model systematically overpredict or underpredict?
    - Is the pattern large enough to matter in the application?


### Mean residuals

- Throughout this notebook:

$$
\text{residual} = \text{actual} - \text{predicted}
$$

- Positive mean residual means the model **underpredicts** for that group.
- Negative mean residual means the model **overpredicts** for that group.
- **Important**: For a linear model with an intercept term, the mean residual across the full **training set** is **always exactly 0**. So, the average residual is not meaningful for the whole dataset, but rather for a subset.

<div class="alert alert-success">

### Discuss: iBudget subgroup performance
    
</div>

- Open: https://apd.myflorida.com/resources/reports/APD%20Algorithm%20Report.pdf
- Find one subgroup or population segment where model performance is discussed. (Don't search for "subgroup" – search for "overestimate".)
- What metrics did they provide? What would you like to have seen?
- In the iBudget example, what are the harms of a model underestimating the true $y_i$'s for a particular subgroup?

### Birthweight subgroup performance


- We'll fit one `Pipeline` on the birthweights dataset together, then inspect its errors by subgroup.
- See if you can understand how the `Pipeline` below is constructed. Can you explain to stakeholders how it works?

In [ ]:
births = load_birthweight_1971()
births = births[[
    "birthweight", "sex", "momrace", "momage", "dadage", "plurality", "birthorder"
]].dropna().copy()

X = births.drop(columns=["birthweight"])
y = births["birthweight"]

X_birth_train, X_birth_test, y_birth_train, y_birth_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=23,
)

In [ ]:
def add_birthweight_features(X):
    X = X.copy() # Ensures modifications aren't made to the original DataFrame.
    X["multiple_birth"] = np.where(X["plurality"] > 1, "yes", "no")
    X["momage_band"] = pd.cut(
        X["momage"],
        bins=[0, 19, 24, 29, 34, 39, 60],
        labels=["under 20", "20-24", "25-29", "30-34", "35-39", "40+"],
    ).astype("object")
    return X

def birthweight_feature_names(transformer, input_features=None):
    if input_features is None:
        input_features = transformer.feature_names_in_
    return np.array(list(input_features) + ["multiple_birth", "momage_band"], dtype=object)

numeric_features = ["momage", "dadage", "birthorder"]
categorical_features = ["sex", "multiple_birth", "momage_band"]

birth_preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    remainder="drop",
)

birth_model = make_pipeline(
    FunctionTransformer(
        add_birthweight_features,
        feature_names_out=birthweight_feature_names,
    ),
    birth_preprocessor,
    LinearRegression(),
)

birth_model.fit(X_birth_train, y_birth_train)

In [ ]:
# Make sure you understand why there are 10 numbers here!
birth_model[-1].coef_

In [ ]:
# model_diagnostics is a helper function defined in setup.py
birth_diagnostics = model_diagnostics(
    birth_model,
    X_birth_train,
    X_birth_test,
    y_birth_train,
    y_birth_test,
)
birth_diagnostics

<div class="alert alert-success">

### Activity: Overall performance
    
</div>

- Before we look at subgroups, write one plain-English sentence summarizing the overall model performance, like you might have seen in the iBudget report.

### Computing residuals

- The code below computes residuals on the **test set**. Remember, the test set is our best guess of what "future data" may look like.

In [ ]:
test_predictions = birth_model.predict(X_birth_test)
report_frame = X_birth_test.copy().assign(
    actual=y_birth_test.to_numpy(),
    predicted=test_predictions,
)
report_frame["residual"] = report_frame["actual"] - report_frame["predicted"]
report_frame.head()

- The helper function below computes various measures of model performance, again on the test set.

In [ ]:
subgroup_metrics(report_frame, group_col="sex")

In [ ]:
# To give you a sense of where the numbers above are coming from.
report_frame.loc[report_frame['sex'] == 'male', 'residual'].abs().mean()

<div class="alert alert-success">

### Discuss: Subgroup performance by maternal race
    
</div>

- Which group has the largest MAE? Smallest?
- Which group has the largest mean residual? Smallest?
- Are the differences large enough to mention in a report?
- Which groups have small enough sample sizes that you would be cautious?

In [ ]:
subgroup_metrics(report_frame, group_col="momrace")

<div class="alert alert-success">

### Discuss: Subgroup performance by plurality
    
</div>

- How much worse is performance for twins or triplet-plus births?
- Is the model overpredicting or underpredicting for any group?
- Why might this happen, given the features in the model?

In [ ]:
report_frame

In [ ]:
subgroup_metrics(report_frame, group_col="plurality").sort_index()

<div class="alert alert-success">

### Activity: Document the model's performance
    
</div>

- Write 1-2 sentences describing the model's overall performance, including its likelihood of generalizing to unseen data.
- State the number of parameters used in the model, and name the parameter with the largest magnitude.
- Write 2-3 sentences describing its performance on various subgroups.
- Ensure that what you write is 

Submit your work to ALICE. **Use the prompt here, rather than what is written in ALICE, as your assessment for Unit 4.4.**

In [ ]:
birth_model[:-1].get_feature_names_out()

### Key takeaways

- Overall performance can hide uneven subgroup performance.
- MAE and RMSE describe error magnitude; mean residual describes direction.
- Positive mean residual means underprediction; negative mean residual means overprediction.
- Subgroup metrics should always be read with group sizes.
- A model report should state what the model can and cannot safely be used for.

### Thanks!

- A few of you have asked for materials from my courses, which dive deeper into some of these topics:
    - For more materials on practical machine learning: [practicaldsc.org](https://practicaldsc.org).
    - Linear algebra and machine learning: [eecs245.org](https://eecs245.org).
- Feel free to reach out: rampure@umich.edu